# 📈 Nifty Technical Analysis & Trading Signals
Welcome to the interactive quickstart notebook. This tutorial walks you through:
1. Loading Nifty 50 market data
2. Computing core technical indicators (Supertrend, RSI, MACD, Bollinger Bands)
3. Generating Buy and Sell signals with dynamic ATR Stop-Loss and Target
4. Running an event-driven backtest and evaluating risk/return metrics

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import fetch_market_data, load_candles_from_csv
from src.data.types import SignalType
from src.strategies.supertrend_rsi import SupertrendRSIStrategy
from src.strategies.confluence import MultiFactorConfluenceStrategy
from src.backtest.engine import BacktestEngine
from src.alerts.notifier import SignalNotifier

print("Engine loaded successfully!")

## 1. Load Market Data
Let's fetch NIFTY 50 (`^NSEI`) daily data.

In [ ]:
candles = fetch_market_data("^NSEI", use_sample_fallback=True)
print(f"Loaded {len(candles)} bars of Nifty 50 data.")
print("First candle:", candles[0].to_dict())
print("Latest candle:", candles[-1].to_dict())

## 2. Generate Signals using Multi-Factor Confluence Strategy
The confluence strategy looks for agreement across Trend (Supertrend/EMA), Momentum (RSI/MACD), Volatility, and Volume.

In [ ]:
strategy = MultiFactorConfluenceStrategy(threshold=0.70)
signals = strategy.generate_signals(candles, symbol="^NSEI")
print(f"Generated {len(signals)} trade signals.\n")

print(SignalNotifier.format_terminal_table(signals[-5:]))

## 3. Backtest Strategy Performance
Run an event-driven backtest with initial capital of ₹100,000 and ₹20 brokerage.

In [ ]:
engine = BacktestEngine(initial_capital=100000.0, brokerage_per_trade=20.0)
result = engine.run(strategy, candles, symbol="^NSEI")

for metric, value in result.summary().items():
    print(f"{metric.ljust(20)}: {value}")